In [0]:
import os

print(os.getcwd())

In [0]:
%sh

cd /Workspace/Users/suraj.shinde3898@outlook.com/Rearc-Assignment-Project

PYTHONPATH=. PYTHONDONTWRITEBYTECODE=1 \
python -m pytest -v -p no:cacheprovider \
tests/test_config_loader.py \
tests/test_population_ingestion.py \
tests/test_bls_ingestion.py \
-k "parser or parse_bls_inventory"

## BLS Change Detection Tests

In [0]:
from datetime import datetime

from src.bls_ingestion import (
    build_comparison_df,
    detect_removed_files,
    build_removed_manifest_records
)

In [0]:
inventory = [
    {
        "file_name": "pr.series",
        "source_url": "https://example.com/pr.series",
        "file_size": 100,
        "source_modified_time": datetime(2026, 8, 6, 8, 30)
    }
]

manifest_schema = """
source string,
file_name string,
file_path string,
file_size long,
source_modified_time timestamp,
ingestion_time timestamp,
status string
"""

empty_manifest_df = spark.createDataFrame(
    [],
    schema=manifest_schema
)

result = (
    build_comparison_df(
        spark,
        inventory,
        empty_manifest_df
    )
    .collect()
)

assert result[0]["action"] == "NEW"

print("PASS: BLS NEW detection")

In [0]:
from src.bls_ingestion import (
    build_comparison_df,
    detect_removed_files,
    build_removed_manifest_records
)
from src.manifest import MANIFEST_SCHEMA

In [0]:
modified_time = datetime(2026, 8, 6, 8, 30)

inventory = [
    {
        "file_name": "pr.series",
        "source_url": "https://example.com/pr.series",
        "file_size": 100,
        "source_modified_time": modified_time
    }
]

manifest_df = spark.createDataFrame(
    [
        (
            "BLS",
            "pr.series",
            "/Volumes/test/pr.series",
            100,
            modified_time,
            datetime(2026, 8, 6, 10, 0),
            "SUCCESS"
        )
    ],
    schema=MANIFEST_SCHEMA
)

result = build_comparison_df(
    spark,
    inventory,
    manifest_df
).collect()

assert result[0]["action"] == "UNCHANGED"

print("PASS: BLS UNCHANGED detection")

In [0]:
inventory = [
    {
        "file_name": "pr.series",
        "source_url": "https://example.com/pr.series",
        "file_size": 200,
        "source_modified_time": modified_time
    }
]

result = build_comparison_df(
    spark,
    inventory,
    manifest_df
).collect()

assert result[0]["action"] == "CHANGED"

print("PASS: BLS CHANGED detection by file size")

In [0]:
inventory = [
    {
        "file_name": "pr.series",
        "source_url": "https://example.com/pr.series",
        "file_size": 100,
        "source_modified_time": datetime(2026, 8, 7, 8, 30)
    }
]

result = build_comparison_df(
    spark,
    inventory,
    manifest_df
).collect()

assert result[0]["action"] == "CHANGED"

print("PASS: BLS CHANGED detection by modified time")

In [0]:
inventory_schema = """
file_name string,
source_url string,
file_size long,
source_modified_time timestamp
"""

empty_inventory_df = spark.createDataFrame(
    [],
    schema=inventory_schema
)

removed_df = detect_removed_files(
    manifest_df,
    empty_inventory_df
)

removed = removed_df.collect()

assert len(removed) == 1
assert removed[0]["file_name"] == "pr.series"

print("PASS: BLS REMOVED detection")

In [0]:
already_removed_df = spark.createDataFrame(
    [
        (
            "BLS",
            "pr.series",
            "/Volumes/test/pr.series",
            100,
            modified_time,
            datetime(2026, 8, 7, 10, 0),
            "REMOVED"
        )
    ],
    schema=MANIFEST_SCHEMA
)

removed_again_df = detect_removed_files(
    already_removed_df,
    empty_inventory_df
)

assert removed_again_df.count() == 0

print("PASS: BLS REMOVED is not duplicated")

## Manifest State Tests

In [0]:
removed_records = build_removed_manifest_records(
    removed_df
)

assert len(removed_records) == 1
assert removed_records[0]["source"] == "BLS"
assert removed_records[0]["file_name"] == "pr.series"
assert removed_records[0]["status"] == "REMOVED"

print("PASS: REMOVED manifest record creation")

In [0]:
from src.manifest import (
    get_latest_manifest_state,
    get_latest_successful_manifest,
    MANIFEST_SCHEMA
)

manifest_test_df = spark.createDataFrame(
    [
        (
            "BLS",
            "pr.series",
            "/Volumes/test/pr.series",
            100,
            datetime(2026, 8, 1, 8, 30),
            datetime(2026, 8, 1, 10, 0),
            "SUCCESS"
        ),
        (
            "BLS",
            "pr.series",
            "/Volumes/test/pr.series",
            100,
            datetime(2026, 8, 1, 8, 30),
            datetime(2026, 8, 2, 10, 0),
            "REMOVED"
        )
    ],
    schema=MANIFEST_SCHEMA
)

temp_view = "test_manifest_latest_state"

manifest_test_df.createOrReplaceTempView(temp_view)

In [0]:
latest_state_df = get_latest_manifest_state(
    spark,
    temp_view,
    "BLS"
)

latest_state = latest_state_df.collect()

assert len(latest_state) == 1
assert latest_state[0]["file_name"] == "pr.series"
assert latest_state[0]["status"] == "REMOVED"

print("PASS: Latest manifest state")

In [0]:
successful_versions_df = spark.createDataFrame(
    [
        (
            "BLS",
            "pr.series",
            "/Volumes/test/pr.series",
            100,
            datetime(2026, 8, 1, 8, 30),
            datetime(2026, 8, 5, 10, 0),
            "SUCCESS"
        ),
        (
            "BLS",
            "pr.series",
            "/Volumes/test/pr.series",
            200,
            datetime(2026, 8, 6, 8, 30),
            datetime(2026, 8, 6, 10, 0),
            "SUCCESS"
        )
    ],
    schema=MANIFEST_SCHEMA
)

successful_view = "test_manifest_successful_versions"

successful_versions_df.createOrReplaceTempView(
    successful_view
)

latest_success_df = get_latest_successful_manifest(
    spark,
    successful_view,
    "BLS"
)

latest_success = latest_success_df.collect()

assert len(latest_success) == 1
assert latest_success[0]["file_size"] == 200
assert latest_success[0]["source_modified_time"] == datetime(
    2026, 8, 6, 8, 30
)

print("PASS: Latest successful BLS version")

In [0]:
mixed_status_df = spark.createDataFrame(
    [
        (
            "BLS",
            "pr.series",
            "/Volumes/test/pr.series",
            100,
            datetime(2026, 8, 1, 8, 30),
            datetime(2026, 8, 1, 10, 0),
            "SUCCESS"
        ),
        (
            "BLS",
            "pr.series",
            "/Volumes/test/pr.series",
            200,
            datetime(2026, 8, 6, 8, 30),
            datetime(2026, 8, 6, 10, 0),
            "FAILED"
        ),
        (
            "BLS",
            "pr.series",
            "/Volumes/test/pr.series",
            100,
            datetime(2026, 8, 1, 8, 30),
            datetime(2026, 8, 7, 10, 0),
            "REMOVED"
        )
    ],
    schema=MANIFEST_SCHEMA
)

mixed_view = "test_manifest_mixed_status"

mixed_status_df.createOrReplaceTempView(
    mixed_view
)

latest_success_df = get_latest_successful_manifest(
    spark,
    mixed_view,
    "BLS"
)

latest_success = latest_success_df.collect()

assert len(latest_success) == 1
assert latest_success[0]["status"] == "SUCCESS"
assert latest_success[0]["file_size"] == 100

print("PASS: FAILED/REMOVED ignored for latest successful version")

In [0]:
multi_source_df = spark.createDataFrame(
    [
        (
            "BLS",
            "pr.series",
            "/Volumes/test/pr.series",
            100,
            datetime(2026, 8, 6, 8, 30),
            datetime(2026, 8, 6, 10, 0),
            "SUCCESS"
        ),
        (
            "POPULATION",
            "population.json",
            "/Volumes/test/population.json",
            500,
            None,
            datetime(2026, 8, 6, 11, 0),
            "SUCCESS"
        )
    ],
    schema=MANIFEST_SCHEMA
)

multi_source_view = "test_manifest_multi_source"

multi_source_df.createOrReplaceTempView(
    multi_source_view
)

bls_state_df = get_latest_manifest_state(
    spark,
    multi_source_view,
    "BLS"
)

rows = bls_state_df.collect()

assert len(rows) == 1
assert rows[0]["source"] == "BLS"
assert rows[0]["file_name"] == "pr.series"

print("PASS: Manifest source filtering")

In [0]:
population_manifest_test_df = spark.createDataFrame(
    [
        {
            "source": "POPULATION",
            "file_name": "population.json",
            "file_path": "/Volumes/test/population.json",
            "file_size": 500,
            "source_modified_time": None,
            "ingestion_time": datetime(2026, 8, 9, 10, 0),
            "status": "SUCCESS"
        }
    ],
    schema=MANIFEST_SCHEMA
)

population_row = population_manifest_test_df.collect()[0]

assert population_row["source_modified_time"] is None

print("PASS: Population manifest supports null source_modified_time")